# BoldSearch query-only: Zilliz + MP4 theo yêu cầu

Notebook này **không** chạy AutoShot, FG-CLIP pipeline, publish corpus, hay ingest Milvus. Nó chỉ query collection Zilliz/Milvus đã tồn tại. Khi trình duyệt cần ảnh của một kết quả, gateway mới decode đúng frame đó từ MP4 và cache WebP trong `/kaggle/working`. Vì vậy Run All không quét hàng nghìn video.

Yêu cầu: bật GPU, attach dataset MP4 có layout `Videos_Lxx_*/video/Lxx_Vnnn.mp4`, attach model FG-CLIP2 offline đầy đủ, và attach Kaggle Input riêng tư chứa file `a.env`. File này phải có `ZILLIZ_URI` và `ZILLIZ_TOKEN`; nó không được in ra output hoặc commit vào Git. Collection phải chứa vector FG-CLIP2 1024 chiều với các field tối thiểu `video_id`, `frame_id`, `shot_id`, `visual_embedding`.

Lưu ý: `Frames.csv` được tạo rỗng có chủ đích. Grid kết quả vẫn hiển thị frame trả về từ query; phần slideshow toàn bộ frame của video không khả dụng vì notebook này không tải/index toàn bộ frame.

In [ ]:
import json
import os
import re
import shutil
import signal
import subprocess
import sys
import time
from pathlib import Path

APP_REPO_URL = 'https://github.com/ToiLaKiet/BoldSearch.git'
APP_REPO_REF = 'main'
RUNTIME_REPO_URL = APP_REPO_URL
# Branch có gateway decode frame trực tiếp từ MP4.
RUNTIME_REPO_REF = 'feat/kaggle-mp4-run-all'

APP_REPO_ROOT = Path('/kaggle/working/BoldSearch')
RUNTIME_REPO_ROOT = Path('/kaggle/working/boldsearch-query-runtime-code')
RUNTIME_ROOT = Path('/kaggle/working/boldsearch-query-runtime')
PUBLIC_ROOT = RUNTIME_ROOT / 'public'
FRONTEND_DIST = PUBLIC_ROOT / 'frontend-dist'
VIDEO_MANIFEST = RUNTIME_ROOT / 'videos.json'
FRAME_CACHE_ROOT = RUNTIME_ROOT / 'frame-cache'

# Đúng tên collection Zilliz/Milvus đã được ingest trước đó. Notebook này không tạo collection mới.
COLLECTION_NAME = 'BoldSearch'
SEARCH_TOP_K = 50
FRAME_MAX_WIDTH = 960
FRAME_WEBP_QUALITY = 82
# Attach một Kaggle Input riêng tư có file này. Đổi tên/path nếu cần.
ENV_INPUT_FILENAME = 'a.env'
ENV_INPUT_PATH_OVERRIDE = None  # ví dụ: Path('/kaggle/input/private-config/a.env')

# Đây là model offline bạn đã attach. Đổi lại nếu Kaggle Input của bạn khác.
FGCLIP_MODEL_PATH_OVERRIDE = Path(
    '/kaggle/input/datasets/quanglongl040305/model2/'
    'aic_l28_offline_models/fgclip2'
)

BACKEND_PORT = 8000
GATEWAY_PORT = 7860

if not Path('/kaggle/input').is_dir() or not Path('/kaggle/working').is_dir():
    raise RuntimeError('Notebook này phải chạy trên Kaggle.')
if not shutil.which('git') or not shutil.which('ffmpeg') or not shutil.which('ffprobe') or not shutil.which('npm'):
    raise RuntimeError('Kaggle image thiếu git, ffmpeg/ffprobe hoặc npm.')
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Hãy bật GPU trong Kaggle Notebook Settings rồi restart session.')
if not FGCLIP_MODEL_PATH_OVERRIDE.is_dir():
    raise RuntimeError(f'Không thấy model FG-CLIP2 offline: {FGCLIP_MODEL_PATH_OVERRIDE}')
if not any(FGCLIP_MODEL_PATH_OVERRIDE.glob('*.safetensors')) and not any(FGCLIP_MODEL_PATH_OVERRIDE.glob('*.bin')):
    raise RuntimeError('Thư mục FG-CLIP2 thiếu weights *.safetensors hoặc *.bin; attach đầy đủ model dataset.')
for path in (RUNTIME_ROOT, PUBLIC_ROOT, FRAME_CACHE_ROOT):
    path.mkdir(parents=True, exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0))
print('Collection query-only:', COLLECTION_NAME)
print('FG-CLIP offline:', FGCLIP_MODEL_PATH_OVERRIDE)


In [ ]:
def clone_or_verify(url: str, ref: str, root: Path, required: list[Path]) -> str:
    if root.exists():
        if not (root / '.git').is_dir():
            raise RuntimeError(f'{root} exists but is not a Git repository.')
        origin = subprocess.check_output(['git', '-C', str(root), 'remote', 'get-url', 'origin'], text=True).strip()
        if origin.rstrip('/') != url.rstrip('/'):
            raise RuntimeError(f'Unexpected origin for {root}: {origin}')
        dirty = subprocess.check_output(['git', '-C', str(root), 'status', '--porcelain', '--untracked-files=no'], text=True).strip()
        if dirty:
            raise RuntimeError(f'Clone has tracked changes; use a fresh Kaggle session:\n{dirty}')
        subprocess.run(['git', '-C', str(root), 'fetch', '--depth', '1', 'origin', ref], check=True)
        subprocess.run(['git', '-C', str(root), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
    else:
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', ref, url, str(root)], check=True)
    missing = [str(path) for path in required if not path.is_file()]
    if missing:
        raise RuntimeError(f'Clone {root} is missing required files:\n' + '\n'.join(missing))
    return subprocess.check_output(['git', '-C', str(root), 'rev-parse', 'HEAD'], text=True).strip()

app_commit = clone_or_verify(APP_REPO_URL, APP_REPO_REF, APP_REPO_ROOT, [
    APP_REPO_ROOT / 'app/backend/pyproject.toml',
    APP_REPO_ROOT / 'app/backend/uv.lock',
    APP_REPO_ROOT / 'app/frontend/package-lock.json',
])
runtime_commit = clone_or_verify(RUNTIME_REPO_URL, RUNTIME_REPO_REF, RUNTIME_REPO_ROOT, [
    RUNTIME_REPO_ROOT / 'boldsearch_integration/fastapi_launcher.py',
    RUNTIME_REPO_ROOT / 'boldsearch_integration/gateway.py',
    RUNTIME_REPO_ROOT / 'boldsearch_integration/video_frames.py',
    RUNTIME_REPO_ROOT / 'boldsearch_integration/vite.runtime.mjs',
])
BACKEND_ROOT = APP_REPO_ROOT / 'app/backend'
FRONTEND_ROOT = APP_REPO_ROOT / 'app/frontend'
print('App commit:', app_commit)
print('Query runtime commit:', runtime_commit)


In [ ]:
video_re = re.compile(r'^L(?P<level>\d{2})_V(?P<number>\d{2,3})\.mp4$')
group_re = re.compile(r'^Videos_L(?P<level>\d{2})_[a-z0-9]+$')
all_videos = sorted(Path('/kaggle/input').glob('**/Videos_L??_*/video/L??_V???.mp4'))
if not all_videos:
    raise RuntimeError('Attach an AIC MP4 dataset containing Videos_Lxx_*/video/Lxx_Vnnn.mp4.')
VIDEO_PATHS: dict[str, Path] = {}
for video in all_videos:
    match = video_re.fullmatch(video.name)
    group = group_re.fullmatch(video.parent.parent.name)
    if not match or not group or match.group('level') != group.group('level'):
        raise RuntimeError(f'Invalid AIC MP4 layout: {video}')
    if video.stem in VIDEO_PATHS:
        raise RuntimeError(f'Duplicate video ID across mounted datasets: {video.stem}')
    VIDEO_PATHS[video.stem] = video.resolve()

VIDEO_MANIFEST.write_text(json.dumps({key: str(value) for key, value in sorted(VIDEO_PATHS.items())}, indent=2) + '\n', encoding='utf-8')
# Gateway cần active release để phục vụ /Frames.csv; để rỗng để không kích hoạt slideshow/quét ảnh hàng loạt.
QUERY_RELEASE_ID = 'query-only'
ACTIVE_RELEASE = PUBLIC_ROOT / 'releases' / QUERY_RELEASE_ID
ACTIVE_RELEASE.mkdir(parents=True, exist_ok=True)
(ACTIVE_RELEASE / 'Frames.csv').write_text('video_id,frame_id,shot_id\n', encoding='utf-8')
(PUBLIC_ROOT / 'active.json').write_text(json.dumps({'schema_version': '1.0', 'release_id': QUERY_RELEASE_ID}) + '\n', encoding='utf-8')
print(f'Found {len(VIDEO_PATHS)} MP4 videos. No video is decoded in this cell.')
print('Video manifest:', VIDEO_MANIFEST)


In [ ]:
_ENV_KEY_RE = re.compile(r'^[A-Za-z_][A-Za-z0-9_]*$')

def load_a_env(path: Path) -> dict[str, str]:
    settings: dict[str, str] = {}
    for line_number, raw_line in enumerate(path.read_text(encoding='utf-8-sig').splitlines(), start=1):
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        key = key.strip().removeprefix('export ').strip()
        if not _ENV_KEY_RE.fullmatch(key):
            raise RuntimeError(f'{path.name}:{line_number} có tên biến không hợp lệ.')
        value = value.strip()
        if len(value) >= 2 and value[0] == value[-1] and value[0] in (chr(34), chr(39)):
            value = value[1:-1]
        settings[key] = value
    missing = [key for key in ('ZILLIZ_URI', 'ZILLIZ_TOKEN') if not settings.get(key)]
    if missing:
        raise RuntimeError(f'{path} thiếu biến bắt buộc: ' + ', '.join(missing))
    return settings

if ENV_INPUT_PATH_OVERRIDE is not None:
    A_ENV_PATH = Path(ENV_INPUT_PATH_OVERRIDE).expanduser().resolve()
else:
    a_env_candidates = sorted({path.resolve() for path in Path('/kaggle/input').glob(f'**/{ENV_INPUT_FILENAME}')})
    if len(a_env_candidates) != 1:
        raise RuntimeError(f'Attach đúng một private Kaggle Input có {ENV_INPUT_FILENAME}; tìm thấy {len(a_env_candidates)} file.')
    A_ENV_PATH = a_env_candidates[0]
if not A_ENV_PATH.is_file():
    raise RuntimeError(f'Không thấy a.env: {A_ENV_PATH}')
A_ENV_SETTINGS = load_a_env(A_ENV_PATH)
ZILLIZ_URI = A_ENV_SETTINGS['ZILLIZ_URI']
ZILLIZ_TOKEN = A_ENV_SETTINGS['ZILLIZ_TOKEN']
print(f'Loaded private config from: {A_ENV_PATH} (values hidden)')

if shutil.which('uv') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'uv'], check=True)
subprocess.run(['uv', 'sync', '--frozen', '--no-dev'], cwd=BACKEND_ROOT, check=True)
BACKEND_PYTHON = BACKEND_ROOT / '.venv/bin/python'
if not BACKEND_PYTHON.is_file():
    raise RuntimeError(f'uv did not create backend Python: {BACKEND_PYTHON}')
subprocess.run(['npm', 'ci', '--ignore-scripts', '--no-audit', '--no-fund'], cwd=FRONTEND_ROOT, check=True)
build_env = os.environ.copy()
build_env.update({'BOLDSEARCH_FRONTEND_ROOT': str(FRONTEND_ROOT), 'BOLDSEARCH_FRONTEND_DIST': str(FRONTEND_DIST)})
subprocess.run(['npm', 'exec', '--', 'vite', 'build', '--config', str(RUNTIME_REPO_ROOT / 'boldsearch_integration/vite.runtime.mjs')], cwd=FRONTEND_ROOT, env=build_env, check=True)
if not (FRONTEND_DIST / 'index.html').is_file():
    raise RuntimeError('Vite did not produce frontend index.html.')
print('Dependencies and frontend build are ready. No MP4 was processed.')


In [ ]:
def stop_owned_process(pid_path: Path, required_tokens: tuple[str, ...]) -> None:
    if not pid_path.is_file():
        return
    try:
        pid = int(pid_path.read_text(encoding='utf-8').strip())
        cmdline = Path(f'/proc/{pid}/cmdline').read_bytes().decode('utf-8', errors='replace').replace('\0', ' ')
        if all(token in cmdline for token in required_tokens):
            os.kill(pid, signal.SIGTERM)
    except (FileNotFoundError, ProcessLookupError, PermissionError, ValueError):
        pass

def write_dotenv(path: Path, values: dict[str, str]) -> None:
    temporary = path.with_name('.env.tmp')
    temporary.write_text('\n'.join(f'{key}={json.dumps(value)}' for key, value in values.items()) + '\n', encoding='utf-8')
    temporary.chmod(0o600)
    os.replace(temporary, path)
    path.chmod(0o600)

write_dotenv(BACKEND_ROOT / '.env', {
    'HOST': '127.0.0.1', 'PORT': str(BACKEND_PORT), 'API_PREFIX': '/api',
    'SYSTEM_NAME': 'BoldSearch query-only',
    'ZILLIZ_URI': ZILLIZ_URI, 'ZILLIZ_TOKEN': ZILLIZ_TOKEN,
    'MILVUS_COLLECTION': COLLECTION_NAME,
    'MILVUS_OUTPUT_FIELDS': 'frame_id,shot_id,video_id,thumbnail',
    'MILVUS_RANKER_WEIGHTS': '1.0', 'SEARCH_TOP_K': str(SEARCH_TOP_K),
    'LOAD_FG_CLIP_ON_STARTUP': 'true', 'FG_CLIP_DEVICE': 'cuda',
    'HF_TOKEN': '',
    'FRAME_IMAGE_URL_TEMPLATE': '/keyframes/{video_id}/{frame_id}.webp',
    'INCLUDE_EMBEDDING_IN_RESPONSE': 'false',
})
runtime_env = os.environ.copy()
runtime_env.update({
    'PYTHONPATH': str(RUNTIME_REPO_ROOT),
    'BOLDSEARCH_FGCLIP_MODEL_PATH': str(FGCLIP_MODEL_PATH_OVERRIDE),
    'TRANSFORMERS_OFFLINE': '1', 'HF_HUB_OFFLINE': '1',
})
backend_log = RUNTIME_ROOT / 'backend.log'
backend_pid = RUNTIME_ROOT / 'backend.pid'
stop_owned_process(backend_pid, ('boldsearch_integration.fastapi_launcher',))
with backend_log.open('wb') as handle:
    backend_process = subprocess.Popen([str(BACKEND_PYTHON), '-m', 'boldsearch_integration.fastapi_launcher', '--app-root', str(BACKEND_ROOT), '--host', '127.0.0.1', '--port', str(BACKEND_PORT)], cwd=BACKEND_ROOT, env=runtime_env, stdin=subprocess.DEVNULL, stdout=handle, stderr=subprocess.STDOUT, start_new_session=True)
backend_pid.write_text(str(backend_process.pid), encoding='utf-8')
from urllib.request import urlopen
health_url = f'http://127.0.0.1:{BACKEND_PORT}/api/health'
deadline = time.monotonic() + 900
while time.monotonic() < deadline:
    if backend_process.poll() is not None:
        break
    try:
        with urlopen(health_url, timeout=5) as response:
            if response.status == 200:
                print('Backend ready:', health_url)
                break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('Backend startup timed out.\n' + backend_log.read_text(encoding='utf-8', errors='replace')[-6000:])
if backend_process.poll() is not None:
    raise RuntimeError('Backend exited.\n' + backend_log.read_text(encoding='utf-8', errors='replace')[-6000:])


In [ ]:
gateway_log = RUNTIME_ROOT / 'gateway.log'
gateway_pid = RUNTIME_ROOT / 'gateway.pid'
stop_owned_process(gateway_pid, ('boldsearch_integration.gateway',))
gateway_env = runtime_env | {
    'BOLDSEARCH_PUBLIC_ROOT': str(PUBLIC_ROOT),
    'BOLDSEARCH_FRONTEND_DIST': str(FRONTEND_DIST),
    'BOLDSEARCH_BACKEND': f'http://127.0.0.1:{BACKEND_PORT}',
    'BOLDSEARCH_GATEWAY_HOST': '127.0.0.1',
    'BOLDSEARCH_GATEWAY_PORT': str(GATEWAY_PORT),
    'BOLDSEARCH_VIDEO_MANIFEST': str(VIDEO_MANIFEST),
    'BOLDSEARCH_FRAME_CACHE_ROOT': str(FRAME_CACHE_ROOT),
    'BOLDSEARCH_FRAME_MAX_WIDTH': str(FRAME_MAX_WIDTH),
    'BOLDSEARCH_FRAME_WEBP_QUALITY': str(FRAME_WEBP_QUALITY),
}
with gateway_log.open('wb') as handle:
    gateway_process = subprocess.Popen([str(BACKEND_PYTHON), '-m', 'boldsearch_integration.gateway'], env=gateway_env, stdin=subprocess.DEVNULL, stdout=handle, stderr=subprocess.STDOUT, start_new_session=True)
gateway_pid.write_text(str(gateway_process.pid), encoding='utf-8')
gateway_health = f'http://127.0.0.1:{GATEWAY_PORT}/api/health'
for _ in range(60):
    try:
        with urlopen(gateway_health, timeout=5) as response:
            if response.status == 200:
                break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError(gateway_log.read_text(encoding='utf-8', errors='replace')[-6000:])

# Smoke test: chỉ decode frame 0 của MỘT video, chứng minh cache theo yêu cầu hoạt động.
sample_video_id = next(iter(sorted(VIDEO_PATHS)))
sample_url = f'http://127.0.0.1:{GATEWAY_PORT}/keyframes/{sample_video_id}/0.webp'
with urlopen(sample_url, timeout=60) as response:
    if response.headers.get_content_type() != 'image/webp' or not response.read(16):
        raise RuntimeError('On-demand MP4 frame endpoint did not return WebP.')
print('Gateway ready:', f'http://127.0.0.1:{GATEWAY_PORT}/')
print('Cached only one smoke-test frame:', sample_url)

sys.path.insert(0, str(RUNTIME_REPO_ROOT))
from boldsearch_integration.tunnel import ensure_cloudflared, start_quick_tunnel
cloudflared = ensure_cloudflared(Path('/kaggle/working/cloudflared'))
tunnel_process, public_url = start_quick_tunnel(cloudflared, gateway_url=f'http://127.0.0.1:{GATEWAY_PORT}', log_path=RUNTIME_ROOT / 'cloudflared.log')
(RUNTIME_ROOT / 'cloudflared.pid').write_text(str(tunnel_process.pid), encoding='utf-8')
print('Open BoldSearch:', public_url)
print('Logs:', backend_log, gateway_log, RUNTIME_ROOT / 'cloudflared.log')
